In [11]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
import os
from dotenv import load_dotenv
load_dotenv()

True

## Tools
### Models can request to call tools that perform tasks such as fetching data from database,searching web,or running code.Tools are pairings of :
#### a) A schema, including the name of tool,a description,and/or argument definitions(often a JSON schema)
#### b) A function or coroutine to exist

In [13]:
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")
# response=model.invoke("Why do parrots talk?")
# response

In [14]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in location {location}"

model_with_tools=model.bind_tools([get_weather])

In [16]:
response=model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool:{tool_call['name']}")
    print(f"Args:{tool_call['args']}")

content='' additional_kwargs={'reasoning_content': "Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. It requires a location, which is Boston here. I'll call the function with location set to Boston.\n", 'tool_calls': [{'id': 'gxedgkc00', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 154, 'total_tokens': 227, 'completion_time': 0.109957321, 'completion_tokens_details': {'reasoning_tokens': 49}, 'prompt_time': 0.009768072, 'prompt_tokens_details': None, 'queue_time': 0.052394615, 'total_time': 0.119725393}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019f300a-636f-7853-a42e-8b86b71d9b96-0' tool_calls=[{'name': 'get_weather', 'args': {'locat

## Tool Execution loops

In [19]:
messages=[{"role":"user","content":"What the weather in Boston?"}]
ai_msg=model_with_tools.invoke(messages)
messages.append(ai_msg)

# Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tools with generated arguments
    tool_result=get_weather.invoke(tool_call)
    messages.append(tool_result)

final_response=model_with_tools.invoke(messages)
print(final_response.text)

The weather in Boston is sunny. ☀️ Let me know if you need more details!


# Messages

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM. Messages are objects that contain:

* **Role** - Identifies the message type (e.g. system, user)
* **Content** - Represents the actual content of the message (like text, images, audio, documents, etc.)
* **Metadata** - Optional fields such as response information, message IDs, and token usage

LangChain provides a standard message type that works across all model providers, ensuring consistent behavior regardless of the model being called.

## Text Prompt:
### Text prompts are strings -ideal for straightforward generayion taskswhere you don't need conversation history
* **Use Text Prompt** - 
  - You have a single standalone request
  - You dont need conversation history
  - You want minimal code complexity

## Message Prompt -
### Aleternatively you can pass a list of messages to the model y providing a list of essage objects
### Message types

* System message - Tells the model how to behave and provide context for interactions
* Human message - Represents user input and interactions with the model
* AI message - Responses generated by the model, including text content, tool calls, and metadata
* Tool message - Represents the outputs of tool calls

## System Message

A SystemMessage represent an initial set of instructions that primes the model's behavior. You can use a system message to set the tone, define the model's role, and establish guidelines for responses.

## Human Message

A HumanMessage represents user input and interactions. They can contain text, images, audio, files, and any other amount of multimodal content.

## AI Message

An AIMessage represents the output of a model invocation. They can include multimodal data, tool calls, and provider-specific metadata that you can later access.

## Tool Message

For models that support tool calling, AI messages can contain tool calls. Tool messages are used to pass the results of a single tool execution back to the model.

In [23]:
from langchain.messages import SystemMessage,HumanMessage,AIMessage
messages=[
    SystemMessage("You are a poetry expert"),
    HumanMessage("Write a poem in artifical intelligence")
]
response=model.invoke(messages)
response

AIMessage(content='<think>\nOkay, the user wants a poem about artificial intelligence. Let me start by brainstorming some key themes related to AI. There\'s the creation aspect, like how AI is built by humans. Then maybe the duality of AI—its potential for good and the concerns it raises. I should include imagery related to technology, circuits, data. Maybe personify AI as a new form of life or consciousness.\n\nI need to think about structure. Maybe four-line stanzas with a rhyme scheme. Let me consider the flow: start with creation, move into capabilities, then the ethical questions, and end with a hopeful or contemplative note. Words like "silicon veins" or "neural threads" could work for imagery. Also, touch on the idea of learning and adaptation in AI. Make sure to balance technical terms with poetic language to keep it accessible. Avoid being too technical but still convey the essence of AI. Maybe use metaphors like "child of circuit and code" to emphasize its artificial nature. 

In [24]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

# 1. The user asks a question requiring real-time data
turn_1_user = HumanMessage(
    content="Can you convert $100 USD to EUR for me?"
)

# 2. The model intercepts this and decides it needs to call a currency converter tool.
# It responds with an AIMessage containing a 'tool_calls' payload.
turn_2_ai = AIMessage(
    content="", # Often empty because the model prefers to call the tool first
    tool_calls=[
        {
            "name": "currency_converter",
            "args": {"amount": 100, "from_currency": "USD", "to_currency": "EUR"},
            "id": "call_usd_to_eur_001"  # Unique ID generated by the model/provider
        }
    ]
)

# 3. Your local Python code executes the actual API call using the arguments above.
# The result from your actual database or external API is packaged into a ToolMessage.
# NOTE: The 'tool_call_id' MUST match the ID requested by the AIMessage.
turn_3_tool = ToolMessage(
    content="{'status': 'success', 'converted_amount': 92.50, 'rate': 0.925}",
    tool_call_id="call_usd_to_eur_001"
)

# 4. The entire list (history) is sent back to the model.
# The model reads the tool output and outputs the final conversational response.
turn_4_final_ai = AIMessage(
    content="Certainly! $100 USD is currently equivalent to approximately 92.50 EUR based on the latest exchange rate of 0.925."
)

# Collecting the entire conversation history chain:
messages = [
    turn_1_user,      # User request
    turn_2_ai,        # Model decides to call tool
    turn_3_tool,      # Your system gives the tool results back to the model
    turn_4_final_ai   # Model translates the raw tool data into standard text
]

# Print out the flow to see how they exist in the payload
for msg in messages:
    print(f"Type: {type(msg).__name__} | Content: {msg.content}")

Type: HumanMessage | Content: Can you convert $100 USD to EUR for me?
Type: AIMessage | Content: 
Type: ToolMessage | Content: {'status': 'success', 'converted_amount': 92.50, 'rate': 0.925}
Type: AIMessage | Content: Certainly! $100 USD is currently equivalent to approximately 92.50 EUR based on the latest exchange rate of 0.925.
